## Initialization

In [ ]:
# Imports

# from math import exp
# from itertools import starmap
from pathlib import Path
from typing import (
    # Callable,
    # TypeVar,
    # Any,
    Literal,
    # overload
)
# from functools import reduce
# import pickle
import re

import matplotlib.pyplot as plt
import pandas as pd
# import numpy as np
# from scipy.optimize import curve_fit
# from scipy.interpolate import interp1d
# from scipy.signal import deconvolve

from data_processing.dot_env import config
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    # ExperimentNeutronData
)
# from data_processing.dataframe_validation import DetectorDataframeColumn
# from data_processing.experiment_data_keys import ExperimentDataKey
# from data_processing.helpers import stop, get_input_with_default
# from data_processing.loading.dataframe_loading import load_parquet_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.slice_fitting import (
#     get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
# from data_processing.types import BimodalBounds, BimodalParams
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
# from data_processing.types import NasaGenerationSettings, WindowType, NeutronWindowSettings
# from data_processing.reporting.plotting import plot_classification
from data_processing.helpers import (
    stop,
    # get_input_with_default,
    # input_experiment_ids,
    get_midpoints_from_min_max_series
)
# from data_processing.loading.window_loading import (
#     load_side_borders, get_neutron_window_paths)
# from data_processing.processing.neutron_window_strategy.strategy_factory import NeutronStrategyFactory
# from data_processing.processing.neutron_window_strategy.abstract_strategy import AbstractNeutronStrategy

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

## Functions

In [ ]:
# def relative_rmse(x: pd.Series | float, x_err: pd.Series | float, y: pd.Series | float, y_err: pd.Series | float) -> pd.Series | float:
def relative_rmse(values: list[tuple[pd.Series | float, pd.Series | float]]) -> pd.Series | float:
    rel_sq_values = [relative_square_error(x, x_err) for x, x_err in values]
    # rel_sq_x = relative_square_error(x, x_err)
    # rel_sq_y = relative_square_error(y, y_err)
    # rel_sq_sum = rel_sq_x + rel_sq_y
    rel_sq_sum = sum(rel_sq_values)
    if isinstance(rel_sq_sum, pd.Series):
        return rel_sq_sum.pow(1./2)
    else:
        return rel_sq_sum ** (1./2)


def relative_square_error(x: pd.Series | float, x_err: pd.Series | float) -> pd.Series | float:
    # divide x_err by x
    # square it
    # return
    rel_err = x_err / x
    if isinstance(rel_err, pd.Series):
        return rel_err.pow(2).fillna(0)
    else:
        return rel_err ** 2

## Run Settings

In [ ]:
exp_id = "ID-423"
bg_id = "ID-479"

phd_data_path = Path(config["NEUTRON_DATA_FOLDER"])
phd_input_path = phd_data_path
phd_output_path = phd_data_path / "output"
# exp_data_filename = f"{exp_id}-n_spectrum.csv"
# exp_data_path = phd_input_path / exp_data_filename
filename_pattern = re.compile(fr"{exp_id}-spectrum-\d\.\d+h-\d\.\d+h.csv")
exp_data_paths = sorted([path for path in phd_input_path.iterdir()
                         if filename_pattern.match(path.name)],
                        key=lambda path: path.stem)
bg_path = phd_input_path / f"{bg_id}-spectrum.csv"
exp_data_paths.insert(0, bg_path)

sim_data_filenames = [
    # "output_sigma_0.5MeV_wresolution.txt",
    "highEstat-sig0.05MeV_A0.5_B0.01_C0.001.csv.npy",   # used in Fig 7b
]
sim_data_combine = False
sim_data_paths = {filename: phd_input_path / filename
                  for filename in sim_data_filenames}

## Loading

### Experimental Data

In [ ]:
exp_dfs = []
for exp_data_path in exp_data_paths:
    exp_df = pd.read_csv(
        exp_data_path,
        usecols=[
            "Light output bin start (MeVee)",
            "Light output bin end (MeVee)",
            "Neutron counts",
            "Gamma counts"
        ]
    )
    exp_df['Neutron count error'] = exp_df['Neutron counts'].pow(1./2)
    exp_df['Gamma count error'] = exp_df['Gamma counts'].pow(1./2)
    exp_dfs.append(exp_df)

In [ ]:
for exp_df in exp_dfs:
    midpoints = get_midpoints_from_min_max_series(
        exp_df["Light output bin start (MeVee)"],
        exp_df["Light output bin end (MeVee)"],
        exp_df.index
    )
    exp_df["Light output (MeVee)"] = midpoints

### Simulation Data

In [ ]:
sim_dfs = {}
for sim_filename, sim_data_path in sim_data_paths.items():
    if sim_filename in ["output_run1.txt", "output_run2.txt"]:
        sim_df = pd.read_csv(sim_data_path, sep="\t", index_col=False)
    else:
        sim_df = pd.read_fwf(sim_data_path)
    sim_df = sim_df[["NPS", "det_pulse (MeVee)"]].copy()
    sim_df.columns = ["Count rate", "Neutron light output (MeVee)"]
    sim_dfs[sim_filename] = sim_df

In [ ]:
sim_l_cut_dfs = {}
bins_lo = exp_df["Light output bin start (MeVee)"]
bins_hi = exp_df["Light output bin end (MeVee)"]
bins = pd.IntervalIndex.from_arrays(bins_lo, bins_hi)
for sim_filename, sim_df in sim_dfs.items():
    light_output_cut = pd.cut(sim_df["Neutron light output (MeVee)"],
                              bins=bins)
    sim_l_cut_dfs[sim_filename] = light_output_cut

In [ ]:
binned_sim_dfs = {}
for sim_filename, sim_l_cut_df in sim_l_cut_dfs.items():
    binned_sim_df = sim_df.groupby(sim_l_cut_df).sum()[["Count rate"]].copy()
    binned_sim_energy_bins = binned_sim_df.index.to_series()
    midpoints = binned_sim_energy_bins.apply(lambda x: x.mid)
    binned_sim_df["Light output (MeVee)"] = midpoints
    binned_sim_dfs[sim_filename] = binned_sim_df

In [ ]:
if sim_data_combine:
    count_series = []
    light_output_series = None
    for sim_filename, binned_sim_df in binned_sim_dfs.items():
        count_series.append(binned_sim_df['Count rate'])
        if light_output_series is None:
            light_output_series = binned_sim_df["Light output (MeVee)"]
    combined_series = pd.concat(count_series, axis=1)
    combined_mean = combined_series.mean(axis=1)
    combined_sd = combined_series.std(axis=1)
    combined_stats_df = pd.DataFrame(
        {
            "Light output (MeVee)": light_output_series,
            "Count rate": combined_mean,
            "Rate error": combined_sd
        }
    )
    binned_sim_dfs = {"combined": combined_stats_df}
else:
    for binned_sim_df in binned_sim_dfs.values():
        binned_sim_df['Rate error'] = binned_sim_df['Count rate'].pow(1./2)

## Normalization

In [ ]:
# counts = exp_df['Neutron counts']
# errors = exp_df['Count error']

# exp_max = counts.max()
# exp_max_idx = counts.idxmax()
# exp_max_error = errors.loc[exp_max_idx]
# rel_square_max_error = (exp_max_error / exp_max) ** 2

# # norm_counts = counts / exp_max
# norm_counts = counts
# exp_df['Counts (normalized)'] = norm_counts

# # rel_norm_errors = relative_rmse(counts, errors, exp_max, exp_max_error)
# # norm_errors = rel_norm_errors * norm_counts
# norm_errors = errors
# exp_df['Error (normalized)'] = norm_errors

n_counts = [exp_df['Neutron counts'] for exp_df in exp_dfs]
n_errors = [exp_df['Neutron count error'] for exp_df in exp_dfs]

exp_n_maxs = [count.max() for count in n_counts]
exp_n_max = max(exp_n_maxs)
exp_n_max_df_index = exp_n_maxs.index(exp_n_max)
exp_n_max_counts = n_counts[exp_n_max_df_index]
exp_n_max_errors = n_errors[exp_n_max_df_index]
exp_n_max_idx = exp_n_max_counts.idxmax()
exp_n_max_error = exp_n_max_errors.loc[exp_n_max_idx]
rel_square_max_n_error = (exp_n_max_error / exp_n_max) ** 2

exp_n_corr_factors = [exp_n_max / this_max for this_max in exp_n_maxs]
exp_n_adjusted_counts = [
    count * factor for count, factor in zip(n_counts, exp_n_corr_factors)]
for exp_df, adj_count in zip(exp_dfs, exp_n_adjusted_counts):
    exp_df['Neutron counts (normalized)'] = adj_count

for this_counts, this_errors, exp_df in zip(n_counts, n_errors, exp_dfs):
    this_max = this_counts.max()
    this_max_idx = this_counts.idxmax()
    this_max_error = this_errors.loc[this_max_idx]
    rel_norm_errors = relative_rmse([
        (this_counts, this_errors),
        (exp_n_max, exp_n_max_error),
        (this_max, this_max_error)
    ])
    norm_errors = rel_norm_errors * this_counts
    exp_df['Neutron count error (normalized)'] = norm_errors

In [ ]:
g_counts = [exp_df['Gamma counts'] for exp_df in exp_dfs]
g_errors = [exp_df['Gamma count error'] for exp_df in exp_dfs]

exp_g_maxs = [count.max() for count in g_counts]
exp_g_max = max(exp_g_maxs)
exp_g_max_df_index = exp_g_maxs.index(exp_g_max)
exp_g_max_counts = g_counts[exp_g_max_df_index]
exp_g_max_errors = g_errors[exp_g_max_df_index]
exp_g_max_idx = exp_g_max_counts.idxmax()
exp_g_max_error = exp_g_max_errors.loc[exp_g_max_idx]
rel_square_max_g_error = (exp_g_max_error / exp_g_max) ** 2

exp_g_corr_factors = [exp_g_max / this_max for this_max in exp_g_maxs]
exp_g_adjusted_counts = [
    count * factor for count, factor in zip(g_counts, exp_g_corr_factors)]
for exp_df, adj_count in zip(exp_dfs, exp_g_adjusted_counts):
    exp_df['Gamma counts (normalized)'] = adj_count

for this_counts, this_errors, exp_df in zip(g_counts, g_errors, exp_dfs):
    this_max = this_counts.max()
    this_max_idx = this_counts.idxmax()
    this_max_error = this_errors.loc[this_max_idx]
    rel_norm_errors = relative_rmse([
        (this_counts, this_errors),
        (exp_n_max, exp_n_max_error),
        (this_max, this_max_error)
    ])
    norm_errors = rel_norm_errors * this_counts
    exp_df['Gamma count error (normalized)'] = norm_errors

In [ ]:
sim_max = 0
sim_max_error = 0
for binned_sim_df in binned_sim_dfs.values():
    counts = binned_sim_df['Count rate']
    errors = binned_sim_df['Rate error']
    this_sim_max = counts.max()
    this_sim_max_idx = counts.idxmax()
    this_sim_max_error = errors[this_sim_max_idx]
    if this_sim_max > sim_max:
        sim_max = this_sim_max
        sim_max_error = this_sim_max_error
sim_factor = exp_n_max / sim_max
# sim_factor_error = relative_rmse(exp_max, exp_max_error, sim_max, sim_max_error)
# rel_square_max_error = (sim_max_error / sim_max) ** 2

for binned_sim_df in binned_sim_dfs.values():
    counts = binned_sim_df['Count rate']
    errors = binned_sim_df['Rate error']
    
    norm_counts = counts * sim_factor
    binned_sim_df['Counts (normalized)'] = norm_counts
    
    # TODO normalize error
    # rel_square_errors = (errors / counts).pow(2).fillna(0)
    # rel_norm_errors = (rel_square_errors + rel_square_max_error).pow(1./2)
    # rel_norm_errors = relative_rmse(counts, errors, sim_factor, sim_factor_error)
    rel_norm_errors = relative_rmse([(counts, errors), (exp_n_max, exp_n_max_error), (sim_max, sim_max_error)])
    norm_errors = rel_norm_errors * norm_counts
    binned_sim_df['Error (normalized)'] = norm_errors

## Saving

In [ ]:
# new_exp_df = exp_df.set_index("Light output (MeVee)", drop=True).get([
#     "Neutron counts", "Neutron count error", "Neutron counts (normalized)", "Neutron error (normalized)",
#     "Gamma counts", "Gamma count error", "Gamma counts (normalized)", "Gamma error (normalized)"
# ])
new_exp_dfs = [exp_df.set_index("Light output (MeVee)", drop=True).get([
    "Neutron counts", "Neutron count error", "Neutron counts (normalized)", "Neutron count error (normalized)",
    "Gamma counts", "Gamma count error", "Gamma counts (normalized)", "Gamma count error (normalized)"
]) for exp_df in exp_dfs]
exp_df_names = ["Background", "Beam", "eCell"]
exp_dfs_to_merge = [
    df.get(
        ["Neutron counts (normalized)",
         "Neutron count error (normalized)",
         "Gamma counts (normalized)",
         "Gamma count error (normalized)"]
    )
    .add_prefix(f"{name} ")
    for name, df in zip(exp_df_names, new_exp_dfs)
]
# TODO merge dfs
exp_normed_df = pd.concat(exp_dfs_to_merge)

new_sim_df = binned_sim_df.set_index("Light output (MeVee)")
# print(new_exp_df.head())
# print(new_sim_df.head())
for name, exp_df in zip(exp_df_names, new_exp_dfs):
    exp_data_output_path = phd_output_path / f"{exp_id}_phd_exp_data_{name.lower()}.csv"
    exp_df.to_csv(exp_data_output_path)

normed_data_output_path = phd_output_path / f"{exp_id}_phd_exp_data_normalized.csv"
sim_data_output_path = phd_output_path / f"{exp_id}_phd_sim_data.csv"
exp_normed_df.to_csv(normed_data_output_path)
new_sim_df.to_csv(sim_data_output_path)

## Plotting

In [ ]:
sim_xys = {}
sim_errors = {}
sim_xys_raw = {}
sim_errors_raw = {}
for sim_filename, binned_sim_df in binned_sim_dfs.items():
    sim_x = binned_sim_df["Light output (MeVee)"].astype(float) * 1000
    sim_y = binned_sim_df["Counts (normalized)"].astype(float)
    sim_raw_y = binned_sim_df["Count rate"].astype(float)
    sim_error = binned_sim_df["Error (normalized)"].astype(float)
    sim_raw_error = binned_sim_df["Rate error"].astype(float)
    
    sim_xys[sim_filename] = (sim_x, sim_y)
    sim_xys_raw[sim_filename] = (sim_x, sim_raw_y)
    sim_errors[sim_filename] = sim_error
    sim_errors_raw[sim_filename] = sim_raw_error

# exp_x = exp_df.index * 1000
exp_y_bg = exp_normed_df["Background Neutron counts (normalized)"].dropna()
exp_error_bg = exp_normed_df["Background Neutron count error (normalized)"].dropna()
exp_y_beam = exp_normed_df["Beam Neutron counts (normalized)"].dropna()
exp_error_beam = exp_normed_df["Beam Neutron count error (normalized)"].dropna()
exp_y_ecell = exp_normed_df["eCell Neutron counts (normalized)"].dropna()
exp_error_ecell = exp_normed_df["eCell Neutron count error (normalized)"].dropna()
# exp_raw_y = exp_df["Neutron counts"]
# exp_raw_y_error = exp_df["Count error"]
exp_g_y_bg = exp_normed_df["Background Gamma counts (normalized)"].dropna()
exp_g_error_bg = exp_normed_df["Background Gamma count error (normalized)"].dropna()
exp_g_y_beam = exp_normed_df["Beam Gamma counts (normalized)"].dropna()
exp_g_error_beam = exp_normed_df["Beam Gamma count error (normalized)"].dropna()
exp_g_y_ecell = exp_normed_df["eCell Gamma counts (normalized)"].dropna()
exp_g_error_ecell = exp_normed_df["eCell Gamma count error (normalized)"].dropna()

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

fig, axs = plt.subplots(1, 1, figsize=(8, 5))
# fig.tight_layout()

for sim_filename, sim_xy in sim_xys.items():
    sim_x, sim_y = sim_xy
    sim_error = sim_errors.get(sim_filename)
    axs.errorbar(
        sim_x,
        sim_y,
        yerr=sim_error,
        ls='-',
        marker='',
        ms=5,
        capsize=2,
        # color='#424242FF',
        label="Simulation"
    )
# axs.errorbar(
#     exp_x,
#     exp_y_bg,
#     yerr=exp_error,
#     ls='--',
#     marker='',
#     ms=5,
#     capsize=2,
#     label="Experiment"
# )
all_y_series = [exp_y_bg, exp_y_beam, exp_y_ecell]
all_y_err_series = [exp_error_bg, exp_error_beam, exp_error_ecell]
labels = ["Background", "Beam-loading only", "Beam-loading + eCell"]
for label, y_series, y_err_series in zip(labels, all_y_series, all_y_err_series):
    x_series = y_series.index * 1000
    axs.errorbar(
        x_series,
        y_series,
        yerr=y_err_series,
        ls='--',
        marker='',
        ms=5,
        capsize=2,
        label=label
    )

# axs.set_title('Beam Loading', fontsize = 16)
axs.set_xlim(-20, 1400)
# axs.set_xlim(200, 240)
# axs.set_ylim(0.99, 1.01)
axs.set_ylabel('Normalized counts', fontsize=fontsize)
# axs.set_yscale("log")
axs.set_xlabel('Light output (keVee)', fontsize=fontsize)
# axs.axhline(64, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(45, color = 'black', ls = "--", alpha = 0.7)
axs.tick_params(axis="x", labelsize=fontsize)
axs.tick_params(axis="y", labelsize=fontsize)
# axs.annotate(
#     'Experiment',
#     (700, 0.15),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14
# )
# axs.annotate(
#     'Simulation',
#     (400, 0.05),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14,
#     color='C0'
# )
axs.legend(
    # loc=(0.78, 0.85)
)
fig.tight_layout()

base_file_name = f"{exp_id} Beam-eCell vs {bg_id} background vs Sim"
fig.savefig(phd_output_path / f"{base_file_name}.png", format="png")
fig.savefig(phd_output_path / f"{base_file_name}.pdf", format="pdf")

plt.show()

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

fig, axs = plt.subplots(
    1, 1,
    figsize=(8, 5)
)

for sim_filename, sim_xy in sim_xys.items():
    sim_x, sim_y = sim_xy
    sim_error = sim_errors.get(sim_filename)
    axs.errorbar(
        sim_x,
        sim_y,
        yerr=sim_error,
        ls='-',
        marker='',
        ms=5,
        capsize=2,
        # color='#424242FF',
        label="Simulation"
    )
# axs.errorbar(
#     exp_x,
#     exp_y,
#     yerr=exp_error,
#     ls='--',
#     marker='',
#     ms=5,
#     capsize=2,
#     label="Experiment"
# )
all_y_series = [exp_y_bg, exp_y_beam, exp_y_ecell]
all_y_err_series = [exp_error_bg, exp_error_beam, exp_error_ecell]
labels = ["Background", "Beam-loading only", "Beam-loading + eCell"]
for label, y_series, y_err_series in zip(labels, all_y_series, all_y_err_series):
    x_series = y_series.index * 1000
    axs.errorbar(
        x_series,
        y_series,
        yerr=y_err_series,
        ls='--',
        marker='',
        ms=5,
        capsize=2,
        label=label
    )

# axs.set_title('Beam Loading', fontsize = 16)
axs.set_xlim(-20, 1400)
axs.set_ylim(1, 100000)
# axs.set_ylim(-5,180)
axs.set_ylabel('Normalized counts', fontsize=fontsize)
axs.set_yscale("log")
axs.set_xlabel('Light output (keVee)', fontsize=fontsize)
# axs.axhline(64, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(45, color = 'black', ls = "--", alpha = 0.7)
axs.tick_params(axis="x", labelsize=fontsize)
axs.tick_params(axis="y", labelsize=fontsize)
# axs.annotate(
#     'Experiment',
#     (700,0.15),
#     xytext = None,
#     xycoords = 'data',
#     textcoords = 'data',
#     fontsize = 14
# )
# axs.annotate(
#     'Simulation',
#     (300, 0.0),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14,
#     color='C0'
# )
# axs.legend(
#     # loc=(0.8, 0.85)
# )
axs.legend()
fig.tight_layout()

base_file_name = f"{exp_id} Beam-eCell vs {bg_id} background vs Sim Log Scale"
fig.savefig(phd_output_path / f"{base_file_name}.png", format="png")
fig.savefig(phd_output_path / f"{base_file_name}.pdf", format="pdf")

plt.show()

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

fig, axs = plt.subplots(1, 1, figsize=(8, 5))
# fig.tight_layout()

# axs.errorbar(
#     exp_x,
#     exp_y_bg,
#     yerr=exp_error,
#     ls='--',
#     marker='',
#     ms=5,
#     capsize=2,
#     label="Experiment"
# )
all_y_series = [exp_g_y_bg, exp_g_y_beam, exp_g_y_ecell]
all_y_err_series = [exp_g_error_bg, exp_g_error_beam, exp_g_error_ecell]
labels = ["Background", "Beam-loading only", "Beam-loading + eCell"]
for label, y_series, y_err_series in zip(labels, all_y_series, all_y_err_series):
    x_series = y_series.index * 1000
    axs.errorbar(
        x_series,
        y_series / 1e6,
        yerr=y_err_series / 1e6,
        ls='--',
        marker='',
        ms=5,
        capsize=2,
        label=label
    )

# axs.set_title('Beam Loading', fontsize = 16)
axs.set_xlim(-20, 1400)
# axs.set_xlim(200, 240)
# axs.set_ylim(0.99, 1.01)
axs.set_ylabel('Normalized counts (x1E6)', fontsize=fontsize)
# axs.set_yscale("log")
axs.set_xlabel('Light output (keVee)', fontsize=fontsize)
# axs.axhline(64, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(45, color = 'black', ls = "--", alpha = 0.7)
axs.tick_params(axis="x", labelsize=fontsize)
axs.tick_params(axis="y", labelsize=fontsize)
# axs.annotate(
#     'Experiment',
#     (700, 0.15),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14
# )
# axs.annotate(
#     'Simulation',
#     (400, 0.05),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14,
#     color='C0'
# )
axs.legend()
fig.tight_layout()

base_file_name = f"{exp_id} Beam-eCell vs {bg_id} background Gamma"
fig.savefig(phd_output_path / f"{base_file_name}.png", format="png")
fig.savefig(phd_output_path / f"{base_file_name}.pdf", format="pdf")

plt.show()

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

fig, axs = plt.subplots(1, 1, figsize=(8, 5))
# fig.tight_layout()

# axs.errorbar(
#     exp_x,
#     exp_y_bg,
#     yerr=exp_error,
#     ls='--',
#     marker='',
#     ms=5,
#     capsize=2,
#     label="Experiment"
# )
all_y_series = [exp_g_y_bg, exp_g_y_beam, exp_g_y_ecell]
all_y_err_series = [exp_g_error_bg, exp_g_error_beam, exp_g_error_ecell]
labels = ["Background", "Beam-loading only", "Beam-loading + eCell"]
for label, y_series, y_err_series in zip(labels, all_y_series, all_y_err_series):
    x_series = y_series.index * 1000
    axs.errorbar(
        x_series,
        y_series,
        yerr=y_err_series,
        ls='--',
        marker='',
        ms=5,
        capsize=2,
        label=label
    )

# axs.set_title('Beam Loading', fontsize = 16)
axs.set_xlim(-20, 1400)
# axs.set_xlim(200, 240)
# axs.set_ylim(0.99, 1.01)
axs.set_ylabel('Normalized counts', fontsize=fontsize)
axs.set_yscale("log")
axs.set_xlabel('Light output (keVee)', fontsize=fontsize)
# axs.axhline(64, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(45, color = 'black', ls = "--", alpha = 0.7)
axs.tick_params(axis="x", labelsize=fontsize)
axs.tick_params(axis="y", labelsize=fontsize)
# axs.annotate(
#     'Experiment',
#     (700, 0.15),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14
# )
# axs.annotate(
#     'Simulation',
#     (400, 0.05),
#     xytext=None,
#     xycoords='data',
#     textcoords='data',
#     fontsize=14,
#     color='C0'
# )
axs.legend()
fig.tight_layout()

base_file_name = f"{exp_id} Beam-eCell vs {bg_id} background Gamma Log Scale"
fig.savefig(phd_output_path / f"{base_file_name}.png", format="png")
fig.savefig(phd_output_path / f"{base_file_name}.pdf", format="pdf")

plt.show()

In [ ]:
input("Processing done, hit Enter to finish")
stop()